In [ ]:
import numpy as np 
import pandas as pd 
import os 
import sys 
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt 
# %matplotlib notebook
sys.path.insert(1,'../../../../arimaclust/')
from arimaclust.models.centroid import ConsistentARMAClustering

# USD

In [ ]:
df_usd = pd.read_csv('data/final_usd_2016_2018.csv',index_col='Unnamed: 0')

In [ ]:
df_usd.drop(columns=['VEF','USD'],inplace=True)

In [ ]:
df_usd_scaled = df_usd

In [ ]:
# scaler = StandardScaler()
# df_usd_scaled = pd.DataFrame(scaler.fit_transform(df_usd),columns=df_usd.columns)

In [ ]:
from sklearn.preprocessing import StandardScaler

X = []
for col in df_usd.columns:
    X.append(df_usd_scaled[col].dropna())
    


In [ ]:
model = ConsistentARMAClustering(k=20, tau=10, max_orders=(3, 3))
model.fit(X=X)

In [ ]:
from sklearn import manifold 

mds = manifold.MDS(
    n_components=2,
    max_iter=3000,
    eps=1e-9,
    random_state=42,
    dissimilarity="precomputed",
    n_jobs=-1,
    normalized_stress="auto",
)
pos = mds.fit(model.D).embedding_


nmds = manifold.MDS(
    n_components=2,
    metric=False,
    max_iter=3000,
    eps=1e-12,
    dissimilarity="precomputed",
    random_state=42,
    n_jobs=-1,
    n_init=1,
    normalized_stress="auto",
)
pos = nmds.fit_transform(model.D, init=pos)

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))
ax.scatter(pos[:,0],pos[:,1],c=model.labels_)
for i in range(df_usd.columns.shape[0]):
    ax.annotate(df_usd.columns[i], (pos[i][0], pos[i][1]),fontsize=8)

In [ ]:
for i in range(20):
    print(i)
    df_usd_scaled[df_usd_scaled.columns[model.labels_==i]].rolling(4).mean().plot()
    plt.show()

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage,to_tree, fcluster
from scipy.spatial.distance import squareform
dist_df = pd.DataFrame(data=model.D, columns=df_usd.columns)
dist_matrix_flat = squareform(dist_df)
Z = linkage(dist_matrix_flat, 'ward', metric='precomputed')
# l = fcluster(Z, t=0.5, criterion='distance')

In [ ]:

fig = plt.figure(figsize=(25, 10))
dn = dendrogram(Z,labels=df_usd.columns)
plt.savefig('dend.png',dpi=200)
plt.show()


In [ ]:
l = model.labels_

In [ ]:
cluster = df_usd.columns[l==l[list(df_usd.columns).index('AED')]]

In [ ]:
df_usd[cluster].rolling(3).mean().plot()

In [ ]:
np.save('matrix_16_18_arma.npy',model.D)

In [ ]:
import pandas as pd 
df_names = pd.read_html('../../../../../table.html')[0]

In [ ]:
df_names[df_names.Code == 'BTC']

In [ ]:
dist_matrix = np.load('matrix_16_18_arma.npy')


In [ ]:
from arimaclust.models.neighbours import NearestNeighbourModel
nn_model = NearestNeighbourModel(alpha=0.1,metric='precomputed')
nn_model.fit(dist_matrix)

In [ ]:
l = np.array(nn_model.labels_)
np.unique(l,return_counts=True)

In [ ]:
cluster = df_usd_scaled.columns[l==l[list(df_usd_scaled.columns).index('EUR')]]
cluster

In [ ]:
df_names[df_names['Code'].isin(cluster)]

In [ ]:
df_usd_scaled[['CHF','EUR']].rolling(3).mean().plot()